# News Modeling

Topic modeling involves **extracting features from document terms** and using
mathematical structures and frameworks like matrix factorization and SVD to generate **clusters or groups of terms** that are distinguishable from each other and these clusters of words form topics or concepts

Topic modeling is a method for **unsupervised classification** of documents, similar to clustering on numeric data

These concepts can be used to interpret the main **themes** of a corpus and also make **semantic connections among words that co-occur together** frequently in various documents

Topic modeling can help in the following areas:
- discovering the **hidden themes** in the collection
- **classifying** the documents into the discovered themes
- using the classification to **organize/summarize/search** the documents

Frameworks and algorithms to build topic models:
- Latent semantic indexing
- Latent Dirichlet allocation
- Non-negative matrix factorization

## Latent Dirichlet Allocation (LDA)
The latent Dirichlet allocation (LDA) technique is a **generative probabilistic model** where each **document is assumed to have a combination of topics** similar to a probabilistic latent semantic indexing model

In simple words, the idea behind LDA is that of two folds:
- each **document** can be described by a **distribution of topics**
- each **topic** can be described by a **distribution of words**

### LDA Algorithm

- 1. For each document, **randomly initialize each word to one of the K topics** (k is chosen beforehand)
- 2. For each document D, go through each word w and compute:
    - **P(T |D)** , which is a proportion of words in D assigned to topic T
    - **P(W |T )** , which is a proportion of assignments to topic T over all documents having the word W
- **Reassign word W with topic T** with probability P(T |D)´ P(W |T ) considering all other words and their topic assignments

![LDA](https://raw.githubusercontent.com/subashgandyer/datasets/main/images/LDA.png)

### Steps
- Install the necessary library
- Import the necessary libraries
- Download the dataset
- Load the dataset
- Pre-process the dataset
    - Stop words removal
    - Email removal
    - Non-alphabetic words removal
    - Tokenize
    - Lowercase
    - BiGrams & TriGrams
    - Lemmatization
- Create a dictionary for the document
- Filter low frequency words
- Create an Index to word dictionary
- Train the Topic Model
- Predict on the dataset
- Evaluate the Topic Model
    - Model Perplexity
    - Topic Coherence
- Visualize the topics

### Install the necessary library

### Import the libraries

In [1]:
import re
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import Phrases, LdaModel
from gensim.models.phrases import Phraser
from gensim.models.coherencemodel import CoherenceModel

# spaCy
import spacy

# Visualization
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
pyLDAvis.enable_notebook()

print('All libraries imported successfully!')

All libraries imported successfully!


### Download the dataset
Dataset: https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json

#### 20-Newsgroups dataset
- 11K newsgroups posts
- 20 news topics

In [2]:
# Download the dataset (run this cell once)
# The file is also available locally as newsgroups.json
# !wget https://raw.githubusercontent.com/subashgandyer/datasets/main/newsgroups.json

### Load the dataset

In [3]:
# Load the JSON dataset
with open('newsgroups.json') as f:
    newsgroups = json.load(f)

# Extract the text content into a list
# The dataset stores documents in a dict keyed by string index
papers = [newsgroups['content'][str(i)] for i in range(len(newsgroups['content']))]

print(f'Number of documents: {len(papers)}')
print(f'Number of categories: {len(newsgroups["target_names"])}')

Number of documents: 11314
Number of categories: 11314


In [4]:
# Preview a sample document
print(papers[0][:500])

From: lerxst@wam.umd.edu (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a m


### Preprocess the data

### Email Removal

In [5]:
def remove_emails(texts):
    """Remove email addresses from a list of documents."""
    return [re.sub(r'\S*@\S*\s?', '', text) for text in texts]

In [6]:
# Apply email removal
data = remove_emails(papers)
print('Emails removed. Sample:')
print(data[0][:300])

Emails removed. Sample:
From: (where's my thing)
Subject: WHAT car is this!?
Nntp-Posting-Host: rac3.wam.umd.edu
Organization: University of Maryland, College Park
Lines: 15

 I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/



### Newline Removal

In [7]:
# Remove newline characters and collapse whitespace
data = [re.sub(r'\s+', ' ', text) for text in data]
print('Newlines removed. Sample:')
print(data[0][:300])

Newlines removed. Sample:
From: (where's my thing) Subject: WHAT car is this!? Nntp-Posting-Host: rac3.wam.umd.edu Organization: University of Maryland, College Park Lines: 15 I was wondering if anyone out there could enlighten me on this car I saw the other day. It was a 2-door sports car, looked to be from the late 60s/ ea


### Single Quotes Removal

In [8]:
# Remove single quotes
data = [re.sub(r"\'", "", text) for text in data]
print('Single quotes removed. Sample:')
print(data[0][:300])

Single quotes removed. Sample:
From: (wheres my thing) Subject: WHAT car is this!? Nntp-Posting-Host: rac3.wam.umd.edu Organization: University of Maryland, College Park Lines: 15 I was wondering if anyone out there could enlighten me on this car I saw the other day. It was a 2-door sports car, looked to be from the late 60s/ ear


### Tokenize
- Create **sent_to_words()** 
    - Use **gensim.utils.simple_preprocess**
    - Use **generator** instead of an usual function

In [9]:
def sent_to_words(sentences):
    """
    Generator that tokenizes each sentence using gensim's simple_preprocess.
    simple_preprocess converts text to lowercase tokens and removes punctuation.
    deacc=True removes accents.
    """
    for sentence in sentences:
        yield simple_preprocess(str(sentence), deacc=True)

In [10]:
# Tokenize all documents
data_words = list(sent_to_words(data))

print(f'Number of tokenized documents: {len(data_words)}')
print(f'Sample tokens: {data_words[0][:20]}')

Number of tokenized documents: 11314
Sample tokens: ['from', 'wheres', 'my', 'thing', 'subject', 'what', 'car', 'is', 'this', 'nntp', 'posting', 'host', 'rac', 'wam', 'umd', 'edu', 'organization', 'university', 'of', 'maryland']


### Stop words Removal
- Extend the stop words corpus with the following words
    - from
    - subject
    - re
    - edu
    - use

In [11]:
# Build a comprehensive stopword list (using spaCy's built-in list + custom additions)
from spacy.lang.en.stop_words import STOP_WORDS as spacy_stopwords

stop_words = set(spacy_stopwords)

# Extend with domain-specific stopwords as instructed
extra_stops = ['from', 'subject', 're', 'edu', 'use']
stop_words.update(extra_stops)

print(f'Total stopwords: {len(stop_words)}')
print(f'Custom stops added: {extra_stops}')

Total stopwords: 329
Custom stops added: ['from', 'subject', 're', 'edu', 'use']


#### remove_stopwords( )

In [12]:
def remove_stopwords(texts):
    """
    Remove stopwords from tokenized documents.
    Also filters out very short tokens (< 3 chars).
    """
    return [
        [word for word in doc if word not in stop_words and len(word) > 2]
        for doc in texts
    ]

In [13]:
# Apply stopword removal
data_words_nostops = remove_stopwords(data_words)
print(f'Sample after stopword removal: {data_words_nostops[0][:20]}')

Sample after stopword removal: ['wheres', 'thing', 'car', 'nntp', 'posting', 'host', 'rac', 'wam', 'umd', 'organization', 'university', 'maryland', 'college', 'park', 'lines', 'wondering', 'enlighten', 'car', 'saw', 'day']


### Bigrams
- Use **gensim.models.Phrases**
- 100 as threshold

In [14]:
# Build Bigram model using gensim Phrases
# threshold=100 means only collocations with a score > 100 are accepted as bigrams
bigram  = Phrases(data_words_nostops, min_count=5, threshold=100)
trigram = Phrases(bigram[data_words_nostops], threshold=100)

# Phraser is a faster, read-only version of Phrases
bigram_mod  = Phraser(bigram)
trigram_mod = Phraser(trigram)

In [15]:
# Preview bigrams in first document
print('Bigrams in document 0:')
print([b for b in bigram_mod[data_words_nostops[0]] if '_' in b])

Bigrams in document 0:
['nntp_posting', 'rac_wam', 'maryland_college']


In [16]:
# Print a sample with bigrams visible (shown in original class work)
print(bigram_mod[data_words_nostops[0]])

['wheres', 'thing', 'car', 'nntp_posting', 'host', 'rac_wam', 'umd', 'organization', 'university', 'maryland_college', 'park', 'lines', 'wondering', 'enlighten', 'car', 'saw', 'day', 'door', 'sports', 'car', 'looked', 'late', 'early', 'called', 'bricklin', 'doors', 'small', 'addition', 'bumper', 'separate', 'rest', 'body', 'know', 'tellme', 'model', 'engine', 'specs', 'years', 'production', 'car', 'history', 'info', 'funky', 'looking', 'car', 'mail', 'thanks', 'brought', 'neighborhood', 'lerxst']


#### make_bigrams( )

In [17]:
def make_bigrams(texts):
    """Apply the bigram model to a list of tokenized documents."""
    return [bigram_mod[doc] for doc in texts]

In [18]:
# Apply bigrams
data_words_bigrams = make_bigrams(data_words_nostops)
print(f'Sample with bigrams: {data_words_bigrams[0][:20]}')

Sample with bigrams: ['wheres', 'thing', 'car', 'nntp_posting', 'host', 'rac_wam', 'umd', 'organization', 'university', 'maryland_college', 'park', 'lines', 'wondering', 'enlighten', 'car', 'saw', 'day', 'door', 'sports', 'car']


### Lemmatization
- Use spacy
    - Download spacy en model (if you have not done that before)
    - Load the spacy model

In [19]:
#! python -m spacy download en_core_web_sm

In [20]:
# Load the spaCy English model
# 'parser' and 'ner' are disabled for speed — we only need lemmatization and POS tagging
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

#### lemmatizaton( )

In [21]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [22]:
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

In [23]:
print(data_lemmatized[:1])

[['s', 'thing', 'car', 'nntp_poste', 'host', 'park', 'line', 'wonder', 'enlighten', 'car', 'see', 'day', 'door', 'sport', 'car', 'look', 'late', 'early', 'call', 'door', 'small', 'addition', 'bumper', 'separate', 'rest', 'body', 'know', 'model', 'engine', 'spec', 'year', 'production', 'car', 'history', 'info', 'funky', 'look', 'car', 'mail', 'thank', 'bring', 'neighborhood', 'lerxst']]


### Create a Dictionary

In [24]:
# Create a Gensim dictionary from the lemmatized tokens
# The dictionary maps each unique word to a numeric ID
id2word = corpora.Dictionary(data_lemmatized)

print(f'Dictionary size before filtering: {len(id2word)}')

Dictionary size before filtering: 49328


### Create Corpus

In [25]:
# Create Bag-of-Words corpus: each document is represented as a list of (word_id, frequency) tuples
corpus = [id2word.doc2bow(text) for text in data_lemmatized]

print(f'Corpus size: {len(corpus)} documents')
print(f'Sample BoW (first 5 pairs of doc 0): {corpus[0][:5]}')

Corpus size: 11314 documents
Sample BoW (first 5 pairs of doc 0): [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)]


### Filter low-frequency words

In [26]:
# Filter extremes:
#   no_below=10  : remove words that appear in fewer than 10 documents
#   no_above=0.5 : remove words that appear in more than 50% of documents
id2word.filter_extremes(no_below=10, no_above=0.5)

print(f'Dictionary size after filtering: {len(id2word)}')

# Re-build corpus after filtering
corpus = [id2word.doc2bow(text) for text in data_lemmatized]
print(f'Corpus re-built: {len(corpus)} documents')

Dictionary size after filtering: 7885
Corpus re-built: 11314 documents


### Create Index 2 word dictionary

In [27]:
# The id2word dictionary already maps numeric IDs to words
# Preview a few mappings
print('Index → Word mappings (first 10):')
for idx in list(id2word.keys())[:10]:
    print(f'  {idx:4d} → {id2word[idx]}')

Index → Word mappings (first 10):
     0 → addition
     1 → body
     2 → bring
     3 → bumper
     4 → call
     5 → car
     6 → day
     7 → door
     8 → early
     9 → engine


In [28]:
# Human-readable view of first document's bag-of-words
print('Decoded BoW for document 0 (first 10 pairs):')
[(id2word[pair[0]], pair[1]) for pair in corpus[0][:10]]

Decoded BoW for document 0 (first 10 pairs):


[('addition', 1),
 ('body', 1),
 ('bring', 1),
 ('bumper', 1),
 ('call', 1),
 ('car', 5),
 ('day', 1),
 ('door', 2),
 ('early', 1),
 ('engine', 1)]

### Build a News Topic Model

#### LdaModel
- **num_topics** : this is the number of topics you need to define beforehand
- **chunksize** : the number of documents to be used in each training chunk
- **alpha** : this is the hyperparameters that affect the sparsity of the topics
- **passess** : total number of training assess

In [29]:
# Train LDA Topic Model
# num_topics=20 matches the 20 news categories in the dataset
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=20,        # 20 latent topics to discover
    chunksize=2000,       # number of documents per training chunk
    alpha='auto',         # learn asymmetric alpha from the data
    passes=10,            # number of full passes through the corpus
    random_state=42       # for reproducibility
)

print('LDA model trained successfully!')
print(f'Number of topics: {lda_model.num_topics}')

LDA model trained successfully!
Number of topics: 20


### Print the Keyword in the 10 topics

In [30]:
# Print the top keywords for all 20 topics
from pprint import pprint
pprint(lda_model.print_topics(num_topics=20, num_words=10))

[(0,
  '0.047*"space" + 0.017*"launch" + 0.014*"earth" + 0.013*"mission" + '
  '0.012*"satellite" + 0.011*"moon" + 0.011*"orbit" + 0.009*"flight" + '
  '0.008*"probe" + 0.008*"rocket"'),
 (1,
  '0.026*"power" + 0.023*"wire" + 0.018*"ground" + 0.017*"circuit" + '
  '0.014*"current" + 0.013*"water" + 0.013*"device" + 0.012*"signal" + '
  '0.012*"input" + 0.011*"low"'),
 (2,
  '0.055*"key" + 0.019*"encryption" + 0.015*"chip" + 0.014*"security" + '
  '0.013*"system" + 0.012*"government" + 0.010*"secure" + 0.010*"clipper_chip" '
  '+ 0.009*"bit" + 0.009*"encrypt"'),
 (3,
  '0.023*"israeli" + 0.016*"attack" + 0.015*"war" + 0.013*"arab" + '
  '0.011*"land" + 0.011*"peace" + 0.010*"state" + 0.010*"article" + '
  '0.008*"kill" + 0.007*"civilian"'),
 (4,
  '0.019*"think" + 0.016*"believe" + 0.013*"exist" + 0.013*"evidence" + '
  '0.013*"claim" + 0.012*"argument" + 0.012*"atheist" + 0.012*"say" + '
  '0.012*"mean" + 0.012*"thing"'),
 (5,
  '0.048*"drive" + 0.021*"sale" + 0.020*"price" + 0.014*"se

## Evaluation of Topic Models
- Model Perplexity
- Topic Coherence

### Model Perplexity

Model perplexity is a measurement of **how well** a **probability distribution** or probability model **predicts a sample**

In [31]:
# Compute Model Perplexity
# Lower (more negative) perplexity = better model
perplexity = lda_model.log_perplexity(corpus)
print(f'Model Perplexity: {perplexity:.4f}')

Model Perplexity: -7.6294


### Topic Coherence
Topic Coherence measures score a single topic by measuring the **degree of semantic similarity** between **high scoring words** in the topic.

In [32]:
# Compute Topic Coherence Score (c_v method)
# Higher coherence = more interpretable topics
coherence_model = CoherenceModel(
    model=lda_model,
    texts=data_lemmatized,
    dictionary=id2word,
    coherence='c_v'
)
coherence_score = coherence_model.get_coherence()
print(f'Topic Coherence Score (c_v): {coherence_score:.4f}')

Topic Coherence Score (c_v): 0.5357


### Visualize the Topic Model
- Use **pyLDAvis**
    - designed to help users **interpret the topics** in a topic model that has been fit to a corpus of text data
    - extracts information from a fitted LDA topic model to inform an interactive web-based visualization

In [33]:
# Prepare the pyLDAvis interactive visualization
# sort_topics=False preserves the original topic numbering
vis = gensimvis.prepare(
    lda_model,
    corpus,
    id2word,
    sort_topics=False
)
vis

/Users/ankitjhurani/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/ankitjhurani/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/ankitjhurani/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/ankitjhurani/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/ankitjhurani/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.085999  0.118276       1        1   2.385495
1      0.139773  0.003461       2        1   1.823187
2      0.034660  0.084798       3        1   3.577739
3     -0.174372  0.097625       4        1   2.488737
4     -0.163418  0.002254       5        1   4.732617
5      0.183755 -0.127602       6        1   3.376027
6      0.045552 -0.183099       7        1   4.590280
7      0.115189  0.132508       8        1   3.975969
8     -0.002853 -0.131590       9        1  10.407955
9      0.136748  0.177214      10        1   6.336208
10    -0.135473  0.031896      11        1   7.091186
11    -0.108432  0.096989      12        1   6.329391
12     0.217060  0.004678      13        1   5.166028
13    -0.160993  0.144336      14        1   5.174858
14    -0.026601  0.023528      15        1   3.650117
15    -0.068544 -0.149893      16        1   2.362290
16     0.155711 -0.042810      17        1   6.949422
17    -0.146411 -0.093377      18        1  13.446729
18    -0.048442  0.039036      19        1   1.713944
19    -0.078910 -0.228227      20        1   4.421821, topic_info=            Term         Freq        Total Category  logprob  loglift
20    nntp_poste  4699.000000  4699.000000  Default  30.0000  30.0000
12          host  4316.000000  4316.000000  Default  29.0000  29.0000
564          key  1973.000000  1973.000000  Default  28.0000  28.0000
291         file  2578.000000  2578.000000  Default  27.0000  27.0000
312        drive  2027.000000  2027.000000  Default  26.0000  26.0000
...          ...          ...          ...      ...      ...      ...
220         come   248.592912  3276.368052  Topic20  -5.1298   0.5399
1479        home   202.414534   834.222187  Topic20  -5.3353   1.7024
38       article   249.968479  6520.332817  Topic20  -5.1243  -0.1427
268          say   209.027432  4252.515626  Topic20  -5.3031   0.1058
198        right   203.120910  3126.369832  Topic20  -5.3318   0.3848

[1437 rows x 6 columns], token_table=      Topic      Freq          Term
term                               
3300      3  0.676964             _
3300     14  0.120609             _
3300     15  0.198421             _
5618     16  0.988699           aaa
4696      7  0.951459  abc_coverage
...     ...       ...           ...
33       18  0.124647          year
33       20  0.162670          year
5247     12  0.063169           zip
5247     13  0.849268           zip
5247     17  0.084225           zip

[4999 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])

In [34]:
# Optionally save the visualization as a standalone HTML file
pyLDAvis.save_html(vis, 'lda_news_topics.html')
print('Visualization saved to lda_news_topics.html')

Visualization saved to lda_news_topics.html
